# INFERENCIA POR SESIÓN

Cuaderno para reproducir el flujo de inferencia casi en tiempo real:
1. Seleccionar el experimento entrenado y una sesión enriched_*.parquet.
2. Ejecutar `run_inference.py`, que aplica el pipeline al instant y guarda las predicciones por ventana.
3. Analizar métricas y visualizar `fatigue_pred` frente a `physical_fatigue_index` para documentar el comportamiento del modelo.

### 1. CONFIGURACIÓN

Ajusta las rutas según el experimento y la sesión que quieras evaluar.

In [8]:
from pathlib import Path

EXPERIMENT_DIR = Path("../data/results/modeling/experiments/runner_id_20260103_180841")
MODEL_NAME = "gradient_boosting"
ENRICHED_PATH = Path("../data/enriched/enriched_D_191003_runTEST__1_186_LPM_2025_10_07_19_04_27_T4_1.parquet")
OUTPUT_PATH = Path("../data/results/modeling/inference/demo_predictions_notebook.parquet")
WINDOW_SECONDS = 3.0
OVERLAP_RATIO = 0.5
PLAYBACK_SPEED = 0.0

EXPERIMENT_DIR, ENRICHED_PATH, OUTPUT_PATH


(PosixPath('../data/results/modeling/experiments/runner_id_20260103_180841'),
 PosixPath('../data/enriched/enriched_D_191003_runTEST__1_186_LPM_2025_10_07_19_04_27_T4_1.parquet'),
 PosixPath('../data/results/modeling/inference/demo_predictions_notebook.parquet'))

### 2. EJECUTA EL PROCEDIMIENTO

El script reutiliza el pipeline guardado y genera predicciones por ventana.

In [9]:
import subprocess
import shlex

cmd = [
    "python",
    "../src/models/run_inference.py",
    "--enriched", str(ENRICHED_PATH),
    "--experiment", str(EXPERIMENT_DIR),
    "--model", MODEL_NAME,
    "--window", str(WINDOW_SECONDS),
    "--overlap", str(OVERLAP_RATIO),
    "--output", str(OUTPUT_PATH),
    "--playback-speed", str(PLAYBACK_SPEED),
]
print("Comando:", " ".join(shlex.quote(part) for part in cmd))

resultado = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", resultado.stdout)
print("STDERR:\n", resultado.stderr)
resultado.check_returncode()


Comando: python ../src/models/run_inference.py --enriched ../data/enriched/enriched_D_191003_runTEST__1_186_LPM_2025_10_07_19_04_27_T4_1.parquet --experiment ../data/results/modeling/experiments/runner_id_20260103_180841 --model gradient_boosting --window 3.0 --overlap 0.5 --output ../data/results/modeling/inference/demo_predictions_notebook.parquet --playback-speed 0.0
STDOUT:
 
STDERR:
 2026-01-08 11:46:39,895 - INFO - Evaluación (physical_fatigue_index por ventana) -> MAE=0.0393 RMSE=0.0513 R2=0.9389
2026-01-08 11:46:39,898 - INFO - [t=  0.00s] pred=0.503 | score=0.482
2026-01-08 11:46:39,898 - INFO - [t=  1.51s] pred=0.461 | score=0.455
2026-01-08 11:46:39,898 - INFO - [t=  3.01s] pred=0.784 | score=0.872
2026-01-08 11:46:39,898 - INFO - [t=  4.51s] pred=0.748 | score=0.853
2026-01-08 11:46:39,898 - INFO - [t=  6.01s] pred=0.410 | score=0.393
2026-01-08 11:46:39,898 - INFO - [t=  7.50s] pred=0.643 | score=0.648
2026-01-08 11:46:39,899 - INFO - [t=  9.00s] pred=0.634 | score=0.648
2

### 3. CARGAR PREDICCIONES Y REVISAR LAS MÉTRICAS

En este paso verificamos el rendimiento del modelo y comprobamos que la inferencia sea consistente con los resultados de entrenamiento.

In [10]:
import pandas as pd
import numpy as np

pred_df = pd.read_parquet(OUTPUT_PATH)
pred_df.head()

,file,source_file,start_s,duration,n_samples,acc_x_centered_mean,acc_x_centered_std,acc_x_centered_mad,acc_x_centered_skew,acc_x_centered_kurt,...,grav_z_skew,grav_z_kurt,jerk_mean,jerk_std,jerk_mad,jerk_skew,hr_mean,spo2_mean,physical_fatigue_index,fatigue_pred
0,clean_D_191003_runTEST__1_186_LPM_2025_10_07_1...,enriched_D_191003_runTEST__1_186_LPM_2025_10_0...,0.000000,2.998114,303,0.004780,1.103568,0.842921,-0.459261,-1.080828,...,0.212542,-0.800377,27.664037,23.753535,8.258768,6.381215,184.000000,97.0,0.482,0.503316
1,clean_D_191003_runTEST__1_186_LPM_2025_10_07_1...,enriched_D_191003_runTEST__1_186_LPM_2025_10_0...,1.509112,2.987912,302,0.105648,1.147697,0.631030,-0.656952,-0.762862,...,0.034088,-0.834277,27.193682,15.519325,9.133999,1.374185,184.000000,97.0,0.455,0.461335
2,clean_D_191003_runTEST__1_186_LPM_2025_10_07_1...,enriched_D_191003_runTEST__1_186_LPM_2025_10_0...,3.008057,2.987914,302,0.103820,1.166606,0.434113,-0.748040,-0.838021,...,-0.169226,-0.938873,36.418305,113.142031,10.954054,10.569759,184.370861,97.0,0.872,0.784105
3,clean_D_191003_runTEST__1_186_LPM_2025_10_07_1...,enriched_D_191003_runTEST__1_186_LPM_2025_10_0...,4.506994,2.987849,302,-0.012717,1.152834,0.432070,-0.596381,-1.203282,...,-0.051694,-1.076936,34.497824,113.049003,9.514189,10.644347,184.870861,97.0,0.853,0.747696
4,clean_D_191003_runTEST__1_186_LPM_2025_10_07_1...,enriched_D_191003_runTEST__1_186_LPM_2025_10_0...,6.005875,2.987914,302,-0.086981,1.100474,0.688061,-0.507027,-1.201130,...,-0.332297,-1.219679,21.857968,11.326361,7.930483,0.461856,185.000000,97.0,0.393,0.410212


In [11]:
print(f"Ventanas totales: {len(pred_df):,}")

Ventanas totales: 55


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

metrics = {}
if "physical_fatigue_index" in pred_df.columns and not pred_df["physical_fatigue_index"].isna().all():
    y_true = pred_df["physical_fatigue_index"].to_numpy()
    y_pred = pred_df["fatigue_pred"].to_numpy()
    metrics = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "R2": r2_score(y_true, y_pred),
    }

{clave: round(valor, 4) for clave, valor in metrics.items()}


{'MAE': 0.0393, 'RMSE': 0.0513, 'R2': 0.9389}

### 4. Visualizaciones

Comparación temporal y análisis de dispersión entre `fatigue_pred` y `physical_fatigue_index`

In [15]:
import plotly.express as px

if "physical_fatigue_index" in pred_df.columns and not pred_df["physical_fatigue_index"].isna().all():
    long_df = pred_df.melt(
        id_vars=["start_s"],
        value_vars=["fatigue_pred", "physical_fatigue_index"],
        var_name="serie",
        value_name="valor",
    )
else:
    long_df = pred_df[["start_s", "fatigue_pred"]].assign(serie="fatigue_pred", valor=pred_df["fatigue_pred"])

fig = px.line(
    long_df,
    x="start_s",
    y="valor",
    color="serie",
    title="Predicción vs. valor real por ventana",
)
fig.update_layout(xaxis_title="Tiempo (s)", yaxis_title="Índice de cansancio físico")
fig.show()



In [18]:
if "physical_fatigue_index" in pred_df.columns and not pred_df["physical_fatigue_index"].isna().all():
    fig = px.scatter(
        pred_df,
        x="physical_fatigue_index",
        y="fatigue_pred",
        title="Dispersión valor real vs. valor predicho",
        labels={"physical_fatigue_index": "Valor real", "fatigue_pred": "Valor predicho"},
    )
    min_val = pred_df["physical_fatigue_index"].min()
    max_val = pred_df["physical_fatigue_index"].max()
    fig.add_shape(
        type="line",
        x0=min_val,
        x1=max_val,
        y0=min_val,
        y1=max_val,
        line=dict(color="gray", dash="dash"),
    )
    fig.update_layout(xaxis_title="Valor real", yaxis_title="Valor predicho")
    fig.show()
else:
    print("No hay score real en el archivo; solo se muestra la serie predicha.")